# 情報数学Ⅲ 第14回

In [ ]:
# 必要なデータファイルを取得
import requests

base_url = "https://raw.githubusercontent.com/logics-of-blue/book-python-stats-2nd/refs/heads/main/book-data/"
filenames = [
    "9-4-1-poisson-regression.csv"
]

for filename in filenames:
    url = base_url + filename
    print(f"Downloading {filename}...")
    response = requests.get(url)
    if response.status_code == 200:
        with open(filename, "wb") as f:
            f.write(response.content)
    else:
        print(f"Failed to download {filename}: {response.status_code}")

# 第9部　一般化線形モデル

## 4章　ポアソン回帰

### 実装：分析の準備

In [ ]:
# 数値計算に使うライブラリ
import numpy as np
import pandas as pd
from scipy import stats
# 表示桁数の設定
pd.set_option('display.precision', 3)
np.set_printoptions(precision=3)

# グラフを描画するライブラリ
from matplotlib import pyplot as plt
import seaborn as sns
sns.set()
# グラフの日本語表記
from matplotlib import rcParams
rcParams['font.family'] = 'sans-serif'
rcParams['font.sans-serif'] = 'Meiryo'

# 統計モデルを推定するライブラリ
import statsmodels.formula.api as smf
import statsmodels.api as sm

In [ ]:
# 表示設定(書籍本文のレイアウトと合わせるためであり、必須ではありません)
np.set_printoptions(linewidth=60)
pd.set_option('display.width', 60)

from matplotlib.pylab import rcParams
rcParams['figure.figsize'] = 8, 4

### 実装：ポアソン分布

#### ポアソン分布の確率質量関数

In [ ]:
# ポアソン分布の確率質量関数
round(stats.poisson.pmf(k=1, mu=2), 3)

In [ ]:
# λ=2のポアソン分布に従う乱数
np.random.seed(1)
stats.poisson.rvs(mu=2, size=5)

In [ ]:
#  λを変化させたポアソン分布の確率質量関数
x = np.arange(0,15,1)
poisson_lambda1 = stats.poisson.pmf(mu=1, k=x)
poisson_lambda2 = stats.poisson.pmf(mu=2, k=x)
poisson_lambda5 = stats.poisson.pmf(mu=5, k=x)

# ポアソン分布の確率質量関数の折れ線グラフ
sns.lineplot(x=x, y=poisson_lambda1, color='black', 
             linestyle='dashed', label='$\lambda=1$')
sns.lineplot(x=x, y=poisson_lambda2, color='black', 
             linestyle='dotted', label='$\lambda=2$')
sns.lineplot(x=x, y=poisson_lambda5, color='black', 
             linestyle='solid', label='$\lambda=5$')

#### ポアソン分布と二項分布の関係

In [ ]:
# pが小さくnが大きい二項分布
p = 0.00000002
n = 100000000
binomial = stats.binom.pmf(n=n, p=p, k=x)

# 二項分布とポアソン分布の比較
sns.lineplot(x=x, y=binomial, color='black', 
             linestyle = 'dotted', 
             label='$np=2$の二項分布')
sns.lineplot(x=x, y=poisson_lambda2, color='gray',
             linestyle='solid', 
             label='$\lambda=2$のポアソン分布')

### 実装：データの読み込み

In [ ]:
# データの読み込み
beer = pd.read_csv('9-4-1-poisson-regression.csv')
print(beer.head(3))

### 実装：ポアソン回帰

In [ ]:
# モデル化
mod_pois = smf.glm('beer_number ~ temperature', beer, 
                   family=sm.families.Poisson()).fit()
mod_pois.summary()

### 実装：ポアソン回帰のモデル選択

In [ ]:
# Nullモデル
mod_pois_null = smf.glm(
    'beer_number ~ 1', data=beer, 
    family=sm.families.Poisson()).fit()

In [ ]:
# AICの比較
print('Nullモデル　　：', round(mod_pois_null.aic, 3))
print('変数入りモデル：', round(mod_pois.aic, 3))

### 実装：ポアソン回帰による予測

#### predict関数を使った予測

In [ ]:
# 説明変数
exp_val_20 = pd.DataFrame({'temperature': [20]})
# 売り上げ個数の予測値
mod_pois.predict(exp_val_20)

#### 推定された係数を使った予測

In [ ]:
beta0 = mod_pois.params[0]
beta1 = mod_pois.params[1]
temperature = 20

round(np.exp(beta0 + beta1 * temperature), 3)

### 実装：ポアソン回帰の回帰曲線

In [ ]:
# 予測値の作成
x_plot = np.arange(0, 37)
pred = mod_pois.predict(pd.DataFrame({'temperature': x_plot}))

# 散布図
sns.scatterplot(x='temperature', y='beer_number',
                data=beer, color='black')
# 回帰曲線を上書き
sns.lineplot(x=x_plot, y=pred, color='black')

### 実装：回帰係数の解釈

In [ ]:
# 気温が1℃のときの販売個数の期待値
exp_val_1 = pd.DataFrame({'temperature': [1]})
pred_1 = mod_pois.predict(exp_val_1)

# 気温が2℃のときの販売個数の期待値
exp_val_2 = pd.DataFrame({'temperature': [2]})
pred_2 = mod_pois.predict(exp_val_2)

# 気温が1℃上がると、販売個数は何倍になるか
round(pred_2 / pred_1, 3)

In [ ]:
# 係数のexpをとる
round(np.exp(mod_pois.params['temperature']), 3)